In [5]:
# 1
#
# ================================================================
# POPULATE CORRIDOR_ID FROM MULTIPART BUFFER
# ================================================================

import arcpy
import os
import time


# ================================================================
# USER SETTINGS
# ================================================================

TARGET_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\CONUS_INTERSTATE_BUFFER_SEGMENTS"

BUFFER_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width.gdb\Interstates_CONUS_Final_10_1_Buffer2000ft"

TARGET_ID_FIELD = "Corridor_ID"


# ================================================================
# START
# ================================================================

start_time = time.time()

print("\n" + "=" * 70)
print("POPULATE CORRIDOR_ID FROM MULTIPART BUFFER")
print("=" * 70)

print(f"\nTarget polygons:")
print(f"  {TARGET_FC}")

print(f"\nMultipart buffer:")
print(f"  {BUFFER_FC}")

print(f"\nTarget ID field: {TARGET_ID_FIELD}")
print(f"Buffer ID field: Corridor_ID")


# ================================================================
# STEP 1/5: VALIDATE INPUTS
# ================================================================

print("\n" + "-" * 70)
print("STEP 1/5: Validating inputs")
print("-" * 70)

if not arcpy.Exists(TARGET_FC):
    raise RuntimeError(f"Target feature class does not exist:\n{TARGET_FC}")

if not arcpy.Exists(BUFFER_FC):
    raise RuntimeError(f"Buffer feature class does not exist:\n{BUFFER_FC}")

target_count = int(
    arcpy.management.GetCount(TARGET_FC)[0]
)

buffer_count = int(
    arcpy.management.GetCount(BUFFER_FC)[0]
)

target_oid = arcpy.Describe(TARGET_FC).OIDFieldName

print(f"Target Object ID field: {target_oid}")
print(f"Target polygons: {target_count:,}")
print(f"Buffer features: {buffer_count:,}")

print("\nInputs validated.")


# ================================================================
# STEP 2/5: PERFORM SPATIAL JOIN
# ================================================================

print("\n" + "-" * 70)
print("STEP 2/5: Performing spatial match")
print("-" * 70)

scratch_gdb = arcpy.env.scratchGDB

temp_join = os.path.join(
    scratch_gdb,
    "Corridor_ID_SpatialJoin"
)

if arcpy.Exists(temp_join):
    arcpy.management.Delete(temp_join)

print("Matching polygons to multipart corridor buffers...")
print("  Match relationship: INTERSECT")
print("  Join operation: ONE_TO_ONE")

arcpy.analysis.SpatialJoin(
    target_features=TARGET_FC,
    join_features=BUFFER_FC,
    out_feature_class=temp_join,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)

print("Spatial match complete.")


# ================================================================
# STEP 3/5: READ BUFFER CORRIDOR_ID
# ================================================================

print("\n" + "-" * 70)
print("STEP 3/5: Reading matched Corridor_ID values")
print("-" * 70)

# Because both the target and buffer already have Corridor_ID,
# ArcGIS creates Corridor_ID_1 for the BUFFER field in the
# temporary Spatial Join output.

lookup = {}

with arcpy.da.SearchCursor(
    temp_join,
    ["TARGET_FID", "Corridor_ID_1"]
) as cursor:

    for target_fid, corridor_id in cursor:
        lookup[target_fid] = corridor_id

print(f"Matched records available: {len(lookup):,}")

matched_non_null = sum(
    1
    for value in lookup.values()
    if value is not None
)

print(f"Non-null Corridor_ID matches: {matched_non_null:,}")

if matched_non_null == 0:
    raise RuntimeError(
        "No non-null Corridor_ID values were found in the "
        "buffer field (Corridor_ID_1)."
    )


# ================================================================
# STEP 4/5: PERMANENTLY UPDATE EXISTING FIELD
# ================================================================

print("\n" + "-" * 70)
print("STEP 4/5: Permanently updating target feature class")
print("-" * 70)

updated = 0
unmatched = 0
processed = 0

with arcpy.da.UpdateCursor(
    TARGET_FC,
    [target_oid, TARGET_ID_FIELD]
) as cursor:

    for row in cursor:

        oid = row[0]

        if oid in lookup and lookup[oid] is not None:

            row[1] = lookup[oid]

            cursor.updateRow(row)

            updated += 1

        else:

            unmatched += 1

        processed += 1

        if processed % 10000 == 0:

            print(
                f"  Processed: "
                f"{processed:,} / "
                f"{target_count:,}"
            )

print("\nUpdate complete.")


# ================================================================
# STEP 5/5: CLEAN UP
# ================================================================

print("\n" + "-" * 70)
print("STEP 5/5: Cleaning temporary data")
print("-" * 70)

if arcpy.Exists(temp_join):
    arcpy.management.Delete(temp_join)

print("Temporary spatial join deleted.")


# ================================================================
# FINAL SUMMARY
# ================================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("COMPLETE")
print("=" * 70)

print(f"\nTarget polygons:       {target_count:,}")
print(f"Updated Corridor_ID:   {updated:,}")
print(f"Unmatched polygons:    {unmatched:,}")

print(f"\nElapsed time: {elapsed / 60:.2f} minutes")

if unmatched > 0:
    print(
        "\nWARNING: Some polygons did not receive a Corridor_ID."
    )
else:
    print(
        "\nAll target polygons received a Corridor_ID."
    )

print("\nThe existing Corridor_ID field was permanently updated.")


POPULATE CORRIDOR_ID FROM MULTIPART BUFFER

Target polygons:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\CONUS_INTERSTATE_BUFFER_SEGMENTS

Multipart buffer:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width.gdb\Interstates_CONUS_Final_10_1_Buffer2000ft

Target ID field: Corridor_ID
Buffer ID field: Corridor_ID

----------------------------------------------------------------------
STEP 1/5: Validating inputs
----------------------------------------------------------------------
Target Object ID field: OBJECTID
Target polygons: 39,121
Buffer features: 317

Inputs validated.

----------------------------------------------------------------------
STEP 2/5: Performing spatial match
----------------------------------------------------------------------
Matching polygons to multipart corridor buffers...
  Match relationship: INTERSECT
  Join operation: ONE_TO_ONE
Spatial match complete.

------------

In [ ]:
# 2

import arcpy
from collections import defaultdict

# =====================================================
# USER INPUT
# =====================================================
ROW_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

SEGMENT_ID_FIELD = "Segment_ID"
WIDTH_FIELD = "ROW_Width_Meters"
AVERAGE_FIELD = "Average_Segment_ROW_Width_Meters"

arcpy.env.overwriteOutput = True

# =====================================================
# STEP 1: CALCULATE AVERAGE WIDTH BY SEGMENT_ID
# =====================================================
print("=" * 70)
print("CALCULATING AVERAGE ROW WIDTH BY SEGMENT_ID")
print("=" * 70)

segment_widths = defaultdict(list)

with arcpy.da.SearchCursor(
    ROW_FC,
    [SEGMENT_ID_FIELD, WIDTH_FIELD]
) as cursor:

    for segment_id, width in cursor:

        if segment_id is not None and width is not None:
            segment_widths[segment_id].append(width)

print(f"Unique Segment_ID values found: {len(segment_widths):,}")

# Calculate averages
segment_averages = {}

for segment_id, widths in segment_widths.items():
    segment_averages[segment_id] = sum(widths) / len(widths)

print("Segment averages calculated.")

# =====================================================
# STEP 2: WRITE AVERAGE BACK TO EVERY ROW
# =====================================================
print()
print("=" * 70)
print("WRITING SEGMENT AVERAGES BACK TO FEATURE CLASS")
print("=" * 70)

updated_count = 0
skipped_count = 0

with arcpy.da.UpdateCursor(
    ROW_FC,
    [
        SEGMENT_ID_FIELD,
        WIDTH_FIELD,
        AVERAGE_FIELD
    ]
) as cursor:

    for segment_id, width, current_average in cursor:

        if segment_id in segment_averages:

            average_width = segment_averages[segment_id]

            cursor.updateRow([
                segment_id,
                width,
                average_width
            ])

            updated_count += 1

        else:
            skipped_count += 1

# =====================================================
# STEP 3: SUMMARY
# =====================================================
print()
print("=" * 70)
print("COMPLETE")
print("=" * 70)
print(f"Unique segments averaged: {len(segment_averages):,}")
print(f"Rows updated:             {updated_count:,}")
print(f"Rows skipped:             {skipped_count:,}")
print("=" * 70)

In [ ]:
# 3

import arcpy

# =====================================================
# USER INPUT
# =====================================================
ROW_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

METERS_FIELD = "Average_Segment_ROW_Width_Meters"
FEET_FIELD = "Average_Segment_ROW_Width_Feet"

# Meters to US survey feet
METERS_TO_US_FEET = 3.28083333333333

# =====================================================
# CHECK FIELDS
# =====================================================
fields = [f.name for f in arcpy.ListFields(ROW_FC)]

if METERS_FIELD not in fields:
    raise ValueError(f"Field not found: {METERS_FIELD}")

if FEET_FIELD not in fields:
    raise ValueError(f"Field not found: {FEET_FIELD}")

# =====================================================
# UPDATE NULL FEET VALUES
# =====================================================
updated_count = 0
skipped_count = 0

with arcpy.da.UpdateCursor(
    ROW_FC,
    [METERS_FIELD, FEET_FIELD]
) as cursor:

    for meters, feet in cursor:

        # Only process records where feet is NULL
        if feet is None:

            # Make sure a meter value exists
            if meters is not None:
                feet_value = meters * METERS_TO_US_FEET

                cursor.updateRow([
                    meters,
                    feet_value
                ])

                updated_count += 1

            else:
                skipped_count += 1

# =====================================================
# SUMMARY
# =====================================================
print("=" * 60)
print("ROW WIDTH FEET UPDATE COMPLETE")
print("=" * 60)
print(f"Records updated: {updated_count}")
print(f"Records skipped (both values NULL): {skipped_count}")
print(f"Conversion factor: {METERS_TO_US_FEET} US survey feet per meter")
print("=" * 60)

In [1]:
# 4
#
# ================================================================
# RECLASSIFY ATTRIBUTE FIELD
#
# Average_Segment_ROW_Width_Feet
#
# RECLASSIFICATION:
#   <= 49.684276706096014                         -> 1
#   > 49.684276706096014 and <= 88.91984637752435 -> 2
#   > 88.91984637752435 and <= 1500              -> 3
#
# The reclassified values are permanently written to the
# specified output field.
# ================================================================

import arcpy
import os
import sys
import time


# ================================================================
# USER INPUTS
# ================================================================

# Input feature class
INPUT_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

# Original field containing the values to reclassify
ORIGINAL_FIELD = "Average_Segment_ROW_Width_Feet"

# New/reclassified field to populate
RECLASS_FIELD = "Average_Segment_ROW_Width_Reclassification"


# ================================================================
# RECLASSIFICATION BREAKS
# ================================================================

BREAK_1 = 49.684276706096014
BREAK_2 = 88.91984637752435
BREAK_3 = 1500


# ================================================================
# START
# ================================================================

print()
print("=" * 70)
print("RECLASSIFY ATTRIBUTE FIELD")
print("=" * 70)

print(f"\nInput feature class:")
print(f"  {INPUT_FC}")

print(f"\nOriginal field:")
print(f"  {ORIGINAL_FIELD}")

print(f"\nReclassified field:")
print(f"  {RECLASS_FIELD}")

print("\nReclassification:")
print(f"  <= {BREAK_1}                         -> 1")
print(f"  > {BREAK_1} and <= {BREAK_2}         -> 2")
print(f"  > {BREAK_2} and <= {BREAK_3}         -> 3")


# ================================================================
# VALIDATION
# ================================================================

if not arcpy.Exists(INPUT_FC):
    print("\nERROR: Input feature class does not exist.")
    sys.exit(1)

fields = {f.name: f for f in arcpy.ListFields(INPUT_FC)}

if ORIGINAL_FIELD not in fields:
    print(f"\nERROR: Original field '{ORIGINAL_FIELD}' does not exist.")
    sys.exit(1)

if RECLASS_FIELD not in fields:
    print(f"\nERROR: Reclassified field '{RECLASS_FIELD}' does not exist.")
    print("Create the field in ArcGIS Pro before running this script.")
    sys.exit(1)


# ================================================================
# PROCESS
# ================================================================

start_time = time.time()

total = int(arcpy.management.GetCount(INPUT_FC)[0])

print(f"\nTotal records: {total:,}")
print("\nPopulating reclassified field...")


count_1 = 0
count_2 = 0
count_3 = 0
count_null = 0
count_outside = 0

with arcpy.da.UpdateCursor(
    INPUT_FC,
    [ORIGINAL_FIELD, RECLASS_FIELD]
) as cursor:

    for i, row in enumerate(cursor, start=1):

        value = row[0]

        # --------------------------------------------------------
        # NULL VALUES
        # --------------------------------------------------------
        if value is None:
            row[1] = None
            count_null += 1

        # --------------------------------------------------------
        # CLASS 1
        # --------------------------------------------------------
        elif value <= BREAK_1:
            row[1] = 1
            count_1 += 1

        # --------------------------------------------------------
        # CLASS 2
        # --------------------------------------------------------
        elif value <= BREAK_2:
            row[1] = 2
            count_2 += 1

        # --------------------------------------------------------
        # CLASS 3
        # --------------------------------------------------------
        elif value <= BREAK_3:
            row[1] = 3
            count_3 += 1

        # --------------------------------------------------------
        # VALUES ABOVE MAXIMUM
        # --------------------------------------------------------
        else:
            row[1] = None
            count_outside += 1

        cursor.updateRow(row)

        # Progress message every 10,000 records
        if i % 10000 == 0 or i == total:
            percent = (i / total) * 100 if total else 100
            print(
                f"  Processed {i:,} / {total:,} "
                f"({percent:6.2f}%)"
            )


# ================================================================
# SUMMARY
# ================================================================

elapsed = time.time() - start_time

print()
print("=" * 70)
print("RECLASSIFICATION COMPLETE")
print("=" * 70)

print(f"\nClass 1: {count_1:,}")
print(f"Class 2: {count_2:,}")
print(f"Class 3: {count_3:,}")
print(f"NULL values: {count_null:,}")
print(f"Values > {BREAK_3}: {count_outside:,}")

print(f"\nTotal records processed: {total:,}")
print(f"Processing time: {elapsed:.2f} seconds")

print("\nField populated:")
print(f"  {RECLASS_FIELD}")

print("\nDone.")
print("=" * 70)


RECLASSIFY ATTRIBUTE FIELD

Input feature class:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width

Original field:
  Average_Segment_ROW_Width_Feet

Reclassified field:
  Average_Segment_ROW_Width_Reclassification

Reclassification:
  <= 49.684276706096014                         -> 1
  > 49.684276706096014 and <= 88.91984637752435         -> 2
  > 88.91984637752435 and <= 1500         -> 3

Total records: 180,821

Populating reclassified field...
  Processed 10,000 / 180,821 (  5.53%)
  Processed 20,000 / 180,821 ( 11.06%)
  Processed 30,000 / 180,821 ( 16.59%)
  Processed 40,000 / 180,821 ( 22.12%)
  Processed 50,000 / 180,821 ( 27.65%)
  Processed 60,000 / 180,821 ( 33.18%)
  Processed 70,000 / 180,821 ( 38.71%)
  Processed 80,000 / 180,821 ( 44.24%)
  Processed 90,000 / 180,821 ( 49.77%)
  Processed 100,000 / 180,821 ( 55.30%)
  Processed 110,000 / 180,821 ( 60.83%)
  Processed 120,000 / 180,821 ( 66

In [2]:
# 5
#
# CALCULATE CORRIDOR ROW WIDTH PERCENTAGES
#
# WORKFLOW:
#   1) Group rows by Corridor_ID
#   2) Examine Average_Segment_ROW_Width_Reclassifcation
#   3) Calculate the percentage of LOW / MEDIUM / HIGH rows
#   4) Write the corridor percentages to EVERY row belonging
#      to that Corridor_ID
#
# RECLASSIFICATION:
#   1 = Low
#   2 = Medium
#   3 = High
#
# NULL / values other than 1, 2, or 3 are excluded from the
# percentage denominator.
#
# Run directly in ArcGIS Pro's Python environment.
# ================================================================

import arcpy
import sys
import time


# ================================================================
# USER INPUTS
# ================================================================

# Input feature class
INPUT_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

# Grouping field
CORRIDOR_ID_FIELD = "Corridor_ID"

# Field containing the 1 / 2 / 3 reclassification
RECLASS_FIELD = "Average_Segment_ROW_Width_Reclassification"

# Output fields
LOW_PERCENT_FIELD = "Corridor_Low_ROW_Width_Percentage"
MEDIUM_PERCENT_FIELD = "Corridor_Medium_ROW_Width_Percentage"
HIGH_PERCENT_FIELD = "Corridor_High_ROW_Width_Percentage"


# ================================================================
# START
# ================================================================

print()
print("=" * 70)
print("CALCULATE CORRIDOR ROW WIDTH PERCENTAGES")
print("=" * 70)

print("\nInput feature class:")
print(f"  {INPUT_FC}")

print("\nFields:")
print(f"  Corridor ID:       {CORRIDOR_ID_FIELD}")
print(f"  Reclassification:  {RECLASS_FIELD}")
print(f"  Low percentage:    {LOW_PERCENT_FIELD}")
print(f"  Medium percentage: {MEDIUM_PERCENT_FIELD}")
print(f"  High percentage:   {HIGH_PERCENT_FIELD}")


# ================================================================
# VALIDATION
# ================================================================

if not arcpy.Exists(INPUT_FC):
    print("\nERROR: Input feature class does not exist.")
    sys.exit(1)

fields = {f.name: f for f in arcpy.ListFields(INPUT_FC)}

required_fields = [
    CORRIDOR_ID_FIELD,
    RECLASS_FIELD,
    LOW_PERCENT_FIELD,
    MEDIUM_PERCENT_FIELD,
    HIGH_PERCENT_FIELD
]

for field_name in required_fields:
    if field_name not in fields:
        print(f"\nERROR: Field '{field_name}' does not exist.")
        sys.exit(1)


# ================================================================
# READ ALL ROWS AND GROUP BY CORRIDOR_ID
# ================================================================

start_time = time.time()

print("\nReading corridor records...")

corridors = {}

with arcpy.da.SearchCursor(
    INPUT_FC,
    [CORRIDOR_ID_FIELD, RECLASS_FIELD]
) as cursor:

    for corridor_id, reclass_value in cursor:

        # Skip records with no Corridor_ID
        if corridor_id is None:
            continue

        if corridor_id not in corridors:
            corridors[corridor_id] = {
                "low": 0,
                "medium": 0,
                "high": 0,
                "classified": 0
            }

        # --------------------------------------------------------
        # Count the reclassification values
        # --------------------------------------------------------

        if reclass_value == 1:
            corridors[corridor_id]["low"] += 1
            corridors[corridor_id]["classified"] += 1

        elif reclass_value == 2:
            corridors[corridor_id]["medium"] += 1
            corridors[corridor_id]["classified"] += 1

        elif reclass_value == 3:
            corridors[corridor_id]["high"] += 1
            corridors[corridor_id]["classified"] += 1


# ================================================================
# CALCULATE PERCENTAGES FOR EACH CORRIDOR
# ================================================================

print(f"\nUnique Corridor_ID values found: {len(corridors):,}")
print("\nCalculating corridor percentages...")

corridor_percentages = {}

for corridor_id, counts in corridors.items():

    total_classified = counts["classified"]

    if total_classified > 0:

        low_pct = (
            counts["low"] / total_classified
        ) * 100.0

        medium_pct = (
            counts["medium"] / total_classified
        ) * 100.0

        high_pct = (
            counts["high"] / total_classified
        ) * 100.0

    else:
        low_pct = None
        medium_pct = None
        high_pct = None

    corridor_percentages[corridor_id] = (
        low_pct,
        medium_pct,
        high_pct
    )


# ================================================================
# WRITE PERCENTAGES BACK TO EVERY ROW
# ================================================================

total_records = int(arcpy.management.GetCount(INPUT_FC)[0])

print("\nWriting corridor percentages back to attribute table...")
print(f"Total records: {total_records:,}")

updated = 0
no_corridor_id = 0
no_classified_values = 0

with arcpy.da.UpdateCursor(
    INPUT_FC,
    [
        CORRIDOR_ID_FIELD,
        LOW_PERCENT_FIELD,
        MEDIUM_PERCENT_FIELD,
        HIGH_PERCENT_FIELD
    ]
) as cursor:

    for i, row in enumerate(cursor, start=1):

        corridor_id = row[0]

        # --------------------------------------------------------
        # No Corridor_ID
        # --------------------------------------------------------

        if corridor_id is None:
            row[1] = None
            row[2] = None
            row[3] = None

            no_corridor_id += 1

        # --------------------------------------------------------
        # Corridor_ID exists
        # --------------------------------------------------------

        elif corridor_id in corridor_percentages:

            low_pct, medium_pct, high_pct = corridor_percentages[
                corridor_id
            ]

            row[1] = low_pct
            row[2] = medium_pct
            row[3] = high_pct

            if low_pct is None:
                no_classified_values += 1

            updated += 1

        cursor.updateRow(row)

        # --------------------------------------------------------
        # Progress
        # --------------------------------------------------------

        if i % 10000 == 0 or i == total_records:

            percent_complete = (
                (i / total_records) * 100
                if total_records
                else 100
            )

            print(
                f"  Updated {i:,} / {total_records:,} "
                f"({percent_complete:6.2f}%)"
            )


# ================================================================
# SUMMARY
# ================================================================

elapsed = time.time() - start_time

print()
print("=" * 70)
print("CORRIDOR PERCENTAGE CALCULATION COMPLETE")
print("=" * 70)

print(f"\nUnique corridors processed: {len(corridors):,}")
print(f"Records updated:            {updated:,}")
print(f"Records without Corridor_ID: {no_corridor_id:,}")
print(f"Corridors with no valid 1/2/3 values: {no_classified_values:,}")

print("\nOutput fields populated:")
print(f"  {LOW_PERCENT_FIELD}")
print(f"  {MEDIUM_PERCENT_FIELD}")
print(f"  {HIGH_PERCENT_FIELD}")

print(f"\nProcessing time: {elapsed:.2f} seconds")

print("\nDone.")
print("=" * 70)


CALCULATE CORRIDOR ROW WIDTH PERCENTAGES

Input feature class:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width

Fields:
  Corridor ID:       Corridor_ID
  Reclassification:  Average_Segment_ROW_Width_Reclassification
  Low percentage:    Corridor_Low_ROW_Width_Percentage
  Medium percentage: Corridor_Medium_ROW_Width_Percentage
  High percentage:   Corridor_High_ROW_Width_Percentage

Reading corridor records...

Unique Corridor_ID values found: 317

Calculating corridor percentages...

Writing corridor percentages back to attribute table...
Total records: 180,821
  Updated 10,000 / 180,821 (  5.53%)
  Updated 20,000 / 180,821 ( 11.06%)
  Updated 30,000 / 180,821 ( 16.59%)
  Updated 40,000 / 180,821 ( 22.12%)
  Updated 50,000 / 180,821 ( 27.65%)
  Updated 60,000 / 180,821 ( 33.18%)
  Updated 70,000 / 180,821 ( 38.71%)
  Updated 80,000 / 180,821 ( 44.24%)
  Updated 90,000 / 180,821 ( 49.77%)
  Updated 100

In [4]:
# 6
# 
# ================================================================
# CALCULATE CORRIDOR WIDTH SCORE
#
# Formula:
#   (Low % / 100 × 1) +
#   (Medium % / 100 × 2) +
#   (High % / 100 × 3)
#
# Result:
#   Corridor_Width_Score = weighted average from 1 to 3
#
# NOTE:
#   All required fields must already exist.
# ================================================================

# ----------------------------------------------------------------
# INPUT
# ----------------------------------------------------------------

# Feature class / table
input_fc = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"


# Existing fields
low_field = "Corridor_Low_ROW_Width_Percentage"
medium_field = "Corridor_Medium_ROW_Width_Percentage"
high_field = "Corridor_High_ROW_Width_Percentage"
score_field = "Corridor_Width_Score"


# ----------------------------------------------------------------
# CALCULATE SCORE
# ----------------------------------------------------------------

print("=" * 70)
print("CALCULATING CORRIDOR WIDTH SCORE")
print("=" * 70)

count = 0

with arcpy.da.UpdateCursor(
    input_fc,
    [low_field, medium_field, high_field, score_field]
) as cursor:

    for row in cursor:

        low = row[0]
        medium = row[1]
        high = row[2]

        # Skip rows where any percentage is NULL
        if low is None or medium is None or high is None:
            continue

        # Weighted average
        score = (
            (low / 100.0 * 1) +
            (medium / 100.0 * 2) +
            (high / 100.0 * 3)
        )

        # Permanently write result
        row[3] = score
        cursor.updateRow(row)

        count += 1


# ----------------------------------------------------------------
# COMPLETE
# ----------------------------------------------------------------

print()
print(f"Rows updated: {count:,}")
print()
print("Formula:")
print("  (Low % / 100 × 1)")
print("+ (Medium % / 100 × 2)")
print("+ (High % / 100 × 3)")
print()
print(f"Results written to: {score_field}")
print("=" * 70)

CALCULATING CORRIDOR WIDTH SCORE

Rows updated: 180,821

Formula:
  (Low % / 100 × 1)
+ (Medium % / 100 × 2)
+ (High % / 100 × 3)

Results written to: Corridor_Width_Score


In [13]:
# 7 
#
# ================================================================
# PERMANENT ONE-TO-MANY JOIN
#
# Excel:
#   One record per unique Corridor_ID
#
# Feature Class:
#   Multiple records can have the same Corridor_ID
#
# The script permanently copies ALL Excel fields into the
# feature class for matching Corridor_ID values.
#
# Corridor_ID is used only as the join field and is NOT copied.
# ================================================================

import arcpy
import os
import sys
import time


# ================================================================
# USER INPUTS
# ================================================================

# Feature class to populate
INPUT_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

# Excel file
EXCEL_FILE = r"C:\Users\KyleSteen.AzureAD\Documents\Task 2.1\Corridors.xlsx"

# Excel sheet name
EXCEL_SHEET = "Corridors"

# Join fields
FC_JOIN_FIELD = "Corridor_ID"
EXCEL_JOIN_FIELD = "Corridor_ID"


# ================================================================
# START
# ================================================================

print()
print("=" * 70)
print("PERMANENT ONE-TO-MANY CORRIDOR JOIN")
print("=" * 70)

print("\nFeature class:")
print(f"  {INPUT_FC}")

print("\nExcel file:")
print(f"  {EXCEL_FILE}")

print("\nExcel sheet:")
print(f"  {EXCEL_SHEET}")

print("\nFeature class join field:")
print(f"  {FC_JOIN_FIELD}")

print("\nExcel join field:")
print(f"  {EXCEL_JOIN_FIELD}")


# ================================================================
# VALIDATION
# ================================================================

print("\n" + "-" * 70)
print("VALIDATING INPUTS")
print("-" * 70)

if not arcpy.Exists(INPUT_FC):
    print("\nERROR: Feature class does not exist.")
    print(INPUT_FC)
    sys.exit(1)

if not os.path.exists(EXCEL_FILE):
    print("\nERROR: Excel file does not exist.")
    print(EXCEL_FILE)
    sys.exit(1)


# ================================================================
# IMPORT EXCEL TO GEODATABASE TABLE
# ================================================================

excel_table = os.path.join(
    arcpy.env.scratchGDB,
    "Corridor_Excel_Table"
)

if arcpy.Exists(excel_table):
    arcpy.management.Delete(excel_table)

print("\nImporting Excel table...")

arcpy.conversion.ExcelToTable(
    EXCEL_FILE,
    excel_table,
    EXCEL_SHEET
)

print("  Excel table imported.")


# ================================================================
# GET EXCEL FIELDS
# ================================================================

excel_fields = {
    field.name: field
    for field in arcpy.ListFields(excel_table)
}

if EXCEL_JOIN_FIELD not in excel_fields:
    print(
        f"\nERROR: Excel join field "
        f"'{EXCEL_JOIN_FIELD}' was not found."
    )
    sys.exit(1)


# ------------------------------------------------
# ALL EXCEL FIELDS EXCEPT THE JOIN FIELD
# ------------------------------------------------

excel_fields_to_copy = [
    field.name
    for field in arcpy.ListFields(excel_table)
    if field.name != EXCEL_JOIN_FIELD
    and field.type not in ["OID", "Geometry"]
]


print("\nExcel fields to copy:")
print(f"  {len(excel_fields_to_copy):,} fields")

for field_name in excel_fields_to_copy:
    print(f"    {field_name}")


# ================================================================
# VALIDATE FEATURE CLASS JOIN FIELD
# ================================================================

fc_fields = {
    field.name: field
    for field in arcpy.ListFields(INPUT_FC)
}

if FC_JOIN_FIELD not in fc_fields:
    print(
        f"\nERROR: Feature class join field "
        f"'{FC_JOIN_FIELD}' was not found."
    )
    sys.exit(1)


# ================================================================
# BUILD EXCEL LOOKUP
# ================================================================

print("\n" + "-" * 70)
print("BUILDING EXCEL LOOKUP")
print("-" * 70)

excel_lookup = {}

search_fields = [
    EXCEL_JOIN_FIELD
] + excel_fields_to_copy

duplicate_ids = set()

with arcpy.da.SearchCursor(
    excel_table,
    search_fields
) as cursor:

    for row in cursor:

        corridor_id = row[0]

        if corridor_id is None:
            continue

        if corridor_id in excel_lookup:
            duplicate_ids.add(corridor_id)

        # Store all Excel values
        excel_lookup[corridor_id] = tuple(row[1:])


# ================================================================
# CHECK DUPLICATE CORRIDOR IDs
# ================================================================

if duplicate_ids:

    print("\nERROR: Duplicate Corridor_ID values found in Excel.")

    print(
        f"Duplicate Corridor_ID values: "
        f"{len(duplicate_ids):,}"
    )

    print("\nExamples:")

    for corridor_id in list(duplicate_ids)[:20]:
        print(f"  {corridor_id}")

    print(
        "\nThe script stopped because the Excel table must "
        "contain one record per Corridor_ID."
    )

    if arcpy.Exists(excel_table):
        arcpy.management.Delete(excel_table)

    sys.exit(1)


print(
    f"\nUnique Corridor_ID values in Excel: "
    f"{len(excel_lookup):,}"
)


# ================================================================
# ADD EXCEL FIELDS TO FEATURE CLASS
# ================================================================

print("\n" + "-" * 70)
print("ADDING EXCEL FIELDS TO FEATURE CLASS")
print("-" * 70)

fc_fields = {
    field.name: field
    for field in arcpy.ListFields(INPUT_FC)
}

fields_added = 0
fields_existing = 0

for field_name in excel_fields_to_copy:

    # ------------------------------------------------------------
    # Field already exists
    # ------------------------------------------------------------

    if field_name in fc_fields:

        print(f"  Existing: {field_name}")
        fields_existing += 1
        continue

    # ------------------------------------------------------------
    # Get source Excel field
    # ------------------------------------------------------------

    source_field = excel_fields[field_name]

    if source_field.type == "String":

        field_type = "TEXT"

        length = source_field.length

        if not length or length < 1:
            length = 255

        arcpy.management.AddField(
            INPUT_FC,
            field_name,
            field_type,
            field_length=length
        )

    elif source_field.type in ["Integer", "SmallInteger"]:

        arcpy.management.AddField(
            INPUT_FC,
            field_name,
            "LONG"
        )

    elif source_field.type in ["Double", "Single"]:

        arcpy.management.AddField(
            INPUT_FC,
            field_name,
            "DOUBLE"
        )

    elif source_field.type == "Date":

        arcpy.management.AddField(
            INPUT_FC,
            field_name,
            "DATE"
        )

    elif source_field.type == "BigInteger":

        arcpy.management.AddField(
            INPUT_FC,
            field_name,
            "BIGINTEGER"
        )

    else:

        print(
            f"\nWARNING: Unsupported field type for "
            f"'{field_name}': {source_field.type}"
        )

        print("  Field will be skipped.")
        continue

    print(f"  Added: {field_name}")
    fields_added += 1


# ================================================================
# REFRESH FEATURE CLASS FIELDS
# ================================================================

fc_fields = {
    field.name: field
    for field in arcpy.ListFields(INPUT_FC)
}

# Only retain fields that actually exist in the FC
fields_to_update = [
    field_name
    for field_name in excel_fields_to_copy
    if field_name in fc_fields
]


# ================================================================
# PERMANENTLY POPULATE FEATURE CLASS
# ================================================================

print("\n" + "-" * 70)
print("POPULATING FEATURE CLASS")
print("-" * 70)

start_time = time.time()

total_records = int(
    arcpy.management.GetCount(INPUT_FC)[0]
)

print(f"\nFeature records: {total_records:,}")

update_fields = [
    FC_JOIN_FIELD
] + fields_to_update

updated_count = 0
no_match_count = 0
null_id_count = 0

with arcpy.da.UpdateCursor(
    INPUT_FC,
    update_fields
) as cursor:

    for i, row in enumerate(cursor, start=1):

        corridor_id = row[0]

        # --------------------------------------------------------
        # NULL Corridor_ID
        # --------------------------------------------------------

        if corridor_id is None:

            for x in range(1, len(row)):
                row[x] = None

            null_id_count += 1

        # --------------------------------------------------------
        # MATCH FOUND
        # --------------------------------------------------------

        elif corridor_id in excel_lookup:

            excel_values = excel_lookup[corridor_id]

            for x, value in enumerate(
                excel_values,
                start=1
            ):
                row[x] = value

            updated_count += 1

        # --------------------------------------------------------
        # NO MATCH
        # --------------------------------------------------------

        else:

            for x in range(1, len(row)):
                row[x] = None

            no_match_count += 1

        cursor.updateRow(row)

        # --------------------------------------------------------
        # Progress
        # --------------------------------------------------------

        if i % 10000 == 0 or i == total_records:

            percent = (
                (i / total_records) * 100
                if total_records
                else 100
            )

            print(
                f"  Processed {i:,} / "
                f"{total_records:,} "
                f"({percent:6.2f}%)"
            )


# ================================================================
# CLEAN UP
# ================================================================

if arcpy.Exists(excel_table):
    arcpy.management.Delete(excel_table)


# ================================================================
# SUMMARY
# ================================================================

elapsed = time.time() - start_time

print()
print("=" * 70)
print("PERMANENT JOIN COMPLETE")
print("=" * 70)

print(f"\nUnique Excel Corridor_IDs: {len(excel_lookup):,}")
print(f"Feature records:           {total_records:,}")
print(f"Records updated:           {updated_count:,}")
print(f"Records with no match:     {no_match_count:,}")
print(f"Records with NULL ID:      {null_id_count:,}")

print(f"\nExcel fields copied:       {len(fields_to_update):,}")
print(f"New fields added:          {fields_added:,}")
print(f"Existing fields reused:    {fields_existing:,}")

print("\nAll matching Excel data has been")
print("permanently written to the feature class.")

print(f"\nProcessing time: {elapsed:.2f} seconds")

print("\nDone.")
print("=" * 70)


PERMANENT ONE-TO-MANY CORRIDOR JOIN

Feature class:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width

Excel file:
  C:\Users\KyleSteen.AzureAD\Documents\Task 2.1\Corridors.xlsx

Excel sheet:
  Corridors

Feature class join field:
  Corridor_ID

Excel join field:
  Corridor_ID

----------------------------------------------------------------------
VALIDATING INPUTS
----------------------------------------------------------------------

Importing Excel table...
  Excel table imported.

Excel fields to copy:
  8 fields
    City_A
    State_A
    City_B
    State_B
    COL_F
    COL_G
    COL_H
    COL_I

----------------------------------------------------------------------
BUILDING EXCEL LOOKUP
----------------------------------------------------------------------

Unique Corridor_ID values in Excel: 320

----------------------------------------------------------------------
ADDING EXCEL FIELDS TO FEATURE 

In [7]:
# 
#
# ================================================================
# EXPORT UNIQUE CORRIDOR SUMMARY TO CSV
#
# For every unique Corridor_ID:
#   - Export only ONE row
#   - Pull the corresponding values for:
#       Corridor_High_ROW_Width_Reclassification
#       Corridor_Medium_ROW_Width_Reclassification
#       Corridor_Low_ROW_Width_Reclassification
#       City_A
#       State_A
#       City_B
#       State_B
#
# All rows with the same Corridor_ID are expected to contain
# identical values for these fields.
#
# OUTPUT:
#   One CSV row per unique Corridor_ID
# ================================================================

import arcpy
import csv
import os
import sys
import time


# ================================================================
# USER INPUTS
# ================================================================

# Input feature class
INPUT_FC = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width"

# Output CSV
OUTPUT_CSV = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_Corridor_Width_Summary.csv"


# ================================================================
# FIELD DEFINITIONS
# ================================================================

CORRIDOR_ID_FIELD = "Corridor_ID"

FIELDS_TO_EXPORT = [
    "Corridor_Width_Score",
    "Corridor_High_ROW_Width_Percentage",
    "Corridor_Medium_ROW_Width_Percentage",
    "Corridor_Low_ROW_Width_Percentage",
    "City_A",
    "State_A",
    "City_B",
    "State_B"
]


# ================================================================
# START
# ================================================================

print()
print("=" * 70)
print("EXPORT UNIQUE CORRIDOR SUMMARY")
print("=" * 70)

print("\nInput feature class:")
print(f"  {INPUT_FC}")

print("\nOutput CSV:")
print(f"  {OUTPUT_CSV}")

print("\nFields being exported:")

for field in FIELDS_TO_EXPORT:
    print(f"  {field}")


# ================================================================
# VALIDATION
# ================================================================

print("\n" + "-" * 70)
print("VALIDATING INPUTS")
print("-" * 70)

if not arcpy.Exists(INPUT_FC):
    print("\nERROR: Input feature class does not exist.")
    print(INPUT_FC)
    sys.exit(1)

fc_fields = {
    field.name: field
    for field in arcpy.ListFields(INPUT_FC)
}

required_fields = [
    CORRIDOR_ID_FIELD
] + FIELDS_TO_EXPORT

for field_name in required_fields:

    if field_name not in fc_fields:

        print(
            f"\nERROR: Required field "
            f"'{field_name}' does not exist."
        )

        sys.exit(1)


# ================================================================
# CREATE OUTPUT DIRECTORY IF NEEDED
# ================================================================

output_folder = os.path.dirname(OUTPUT_CSV)

if output_folder and not os.path.exists(output_folder):

    print("\nCreating output directory:")
    print(f"  {output_folder}")

    os.makedirs(output_folder)


# ================================================================
# READ UNIQUE CORRIDORS
# ================================================================

print("\n" + "-" * 70)
print("READING CORRIDOR DATA")
print("-" * 70)

start_time = time.time()

unique_corridors = {}

search_fields = [
    CORRIDOR_ID_FIELD
] + FIELDS_TO_EXPORT

total_records = int(
    arcpy.management.GetCount(INPUT_FC)[0]
)

print(f"\nTotal feature records: {total_records:,}")

with arcpy.da.SearchCursor(
    INPUT_FC,
    search_fields
) as cursor:

    for i, row in enumerate(cursor, start=1):

        corridor_id = row[0]

        # --------------------------------------------------------
        # Skip NULL Corridor_ID
        # --------------------------------------------------------

        if corridor_id is None:
            continue

        # --------------------------------------------------------
        # Only store the FIRST row encountered for each
        # Corridor_ID
        # --------------------------------------------------------

        if corridor_id not in unique_corridors:

            unique_corridors[corridor_id] = tuple(row[1:])

        # --------------------------------------------------------
        # Progress
        # --------------------------------------------------------

        if i % 10000 == 0 or i == total_records:

            percent = (
                (i / total_records) * 100
                if total_records
                else 100
            )

            print(
                f"  Scanned {i:,} / "
                f"{total_records:,} "
                f"({percent:6.2f}%)"
            )


# ================================================================
# SORT CORRIDORS
# ================================================================

print("\nUnique Corridor_ID values found:")
print(f"  {len(unique_corridors):,}")

# Sort by Corridor_ID when possible
try:
    unique_corridors = dict(
        sorted(unique_corridors.items())
    )
except TypeError:
    # If mixed ID types prevent sorting,
    # retain the original order.
    pass


# ================================================================
# WRITE CSV
# ================================================================

print("\n" + "-" * 70)
print("WRITING CSV")
print("-" * 70)

csv_headers = [
    CORRIDOR_ID_FIELD
] + FIELDS_TO_EXPORT

with open(
    OUTPUT_CSV,
    "w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:

    writer = csv.writer(csv_file)

    # Header
    writer.writerow(csv_headers)

    # One row per unique Corridor_ID
    for corridor_id, values in unique_corridors.items():

        writer.writerow(
            [corridor_id] + list(values)
        )


# ================================================================
# SUMMARY
# ================================================================

elapsed = time.time() - start_time

print()
print("=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print(f"\nInput records scanned:       {total_records:,}")
print(f"Unique corridors exported:   {len(unique_corridors):,}")

print("\nCSV columns:")
for field in csv_headers:
    print(f"  {field}")

print("\nOutput:")
print(f"  {OUTPUT_CSV}")

print(f"\nProcessing time: {elapsed:.2f} seconds")

print("\nDone.")
print("=" * 70)


EXPORT UNIQUE CORRIDOR SUMMARY

Input feature class:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_ROW_Width_Results.gdb\AGCC_Final_ROWs_CONUS_ROW_Width

Output CSV:
  C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\CONUS_ROW_Width\CONUS_Corridor_Width_Summary.csv

Fields being exported:
  Corridor_Width_Score
  Corridor_High_ROW_Width_Percentage
  Corridor_Medium_ROW_Width_Percentage
  Corridor_Low_ROW_Width_Percentage
  City_A
  State_A
  City_B
  State_B

----------------------------------------------------------------------
VALIDATING INPUTS
----------------------------------------------------------------------

----------------------------------------------------------------------
READING CORRIDOR DATA
----------------------------------------------------------------------

Total feature records: 180,821
  Scanned 10,000 / 180,821 (  5.53%)
  Scanned 20,000 / 180,821 ( 11.06%)
  Scanned 30,000 / 180,821 ( 16.59%)
  Scanned 40,000 / 180,821 ( 22.